In [ ]:
# DFU Phase-5.1 Final Audit — one cell
import urllib.request, hashlib

VERSION = "DFU_PHASE5_1_FINAL_AUDIT_LOADER_V1_20260812"
SOURCE_COMMIT = "f0d4a50ca444d31907f6505dcdcb7c81d331b762"
SOURCE_PATH = "scripts/phase5_1_final_audit_v1.py"
EXPECTED_GIT_BLOB = "5f28d7ed0d54f0d3db378487852b036f1f6bb2ca"
URL = f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/{SOURCE_PATH}"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

print("="*100)
print(VERSION)
print("EXISTING ARTIFACTS ONLY | NO TRAINING | NO DATASET CNN INFERENCE")
print("="*100)

raw = urllib.request.urlopen(URL, timeout=120).read()
actual_blob = git_blob_sha(raw)
if actual_blob != EXPECTED_GIT_BLOB:
    raise RuntimeError(f"Source Git blob mismatch: expected={EXPECTED_GIT_BLOB} actual={actual_blob}")
print("Pinned source Git blob: PASS", actual_blob)
print("Source bytes:", len(raw))
print("Source SHA256:", hashlib.sha256(raw).hexdigest())

text = raw.decode("utf-8")
for forbidden in ["optimizer.step", ".backward(", "dataset_download(", "timm.create_model", "DataLoader("]:
    if forbidden in text:
        raise RuntimeError(f"Forbidden training/inference token detected: {forbidden}")
print("Static no-training/no-loader safety scan: PASS")

compile(text, "dfu_phase5_1_final_audit_v1.py", "exec")
print("Source compile: PASS")
print("Starting Phase-5.1 final audit...")
exec(compile(text, "dfu_phase5_1_final_audit_v1.py", "exec"), globals())
